In [1]:
!pip install sentence-transformers faiss-cpu
!pip install transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.4 MB/s eta 0:00:00


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Use an open-access model (No token/gate required)
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!


In [3]:
documents = [
    "RAG stands for Retrieval Augmented Generation.",
    "FAISS is a library for efficient similarity search.",
    "Embeddings convert text into vectors.",
    "Chunking is the process of splitting text into smaller parts.",
    "Large Language Models generate text based on input context."
]


In [4]:
chunk_size = 100 ## chunk size defines how many chunks are made
overlap = 25 ## overlap is used to make sure each chunk can be related to the the chunk to the back of it

def chunk_text(text, chunk_size, overlap):
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    step = chunk_size - overlap
    chunks = []

    for i in range(0, len(text), step):
        chunk = text[i:i + chunk_size]
        if chunk:
            chunks.append(chunk)

    return chunks

chunks = []

for doc in documents:
    chunks.extend(chunk_text(doc, chunk_size, overlap))

print(f"Total chunks created: {len(chunks)}")
print(chunks[:3])

Total chunks created: 5
['RAG stands for Retrieval Augmented Generation.', 'FAISS is a library for efficient similarity search.', 'Embeddings convert text into vectors.']


In [5]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

embeddings = embed_model.encode(
    chunks,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [6]:
import faiss
import numpy as np

embeddings = np.array(embeddings).astype("float32")
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension) # Use inner product (cosine similarity if normalized)
index.add(embeddings)
chunk_store = chunks
print("done")

done


In [7]:
def retrieve(query, k=3):
    query = "Represent this sentence for searching: " + query
    query_vec = embed_model.encode(
        [query],
        normalize_embeddings=True
    )

    query_vec = np.array(query_vec).astype("float32")

    distances, indices = index.search(query_vec, k)

    results = [
        {"text": chunks[i], "score": float(distances[0][j])}
        for j, i in enumerate(indices[0])
    ]

    return results
print("done")

done


In [8]:
def build_prompt(query, context):
    return f"""<|system|>
Answer the question using ONLY the context provided.</s>
<|user|>
Context: {context}
Question: {query}</s>
<|assistant|>
"""

In [9]:
def generate_answer(query, retrieved_chunks):
    # Extract text from chunks
    context = "\n\n".join([c["text"] for c in retrieved_chunks])

    prompt = build_prompt(query, context)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=True,
            repetition_penalty=1.1
        )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = full_output.split("<|assistant|>")[-1].strip()

    return answer

In [10]:
def rag_pipeline(query):
    retrieved = retrieve(query)
    answer = generate_answer(query, retrieved)

    return {
        "query": query,
        "retrieved_chunks": retrieved,
        "answer": answer
    }

In [11]:
result = rag_pipeline("What is RAG?")
print(result["answer"])

Response: RAG (Retrieval Augmented Generation) is a technique used in NLP to split text into smaller parts called "chunks" for efficient similarity search. FAISS is a library for efficient similarity search that uses RAG.
